<a href="https://colab.research.google.com/github/prodigal94/food-safe-bots/blob/main/Week_3_Linear_Regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql import SparkSession
from pyspark.ml.regression import LinearRegression
from pyspark.sql.functions import log

In [18]:
# Initialize Spark with optimizations for large data
spark = SparkSession.builder \
    .appName("BDA_Project") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "8g") \
    .getOrCreate()

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [23]:
df = spark.read.parquet("/content/drive/MyDrive/Parquet/02_AgriTrade_ValueOnly.parquet")
df = df.withColumn("logValue", log(df["Value"]))

In [24]:
train, test = df.randomSplit([.8,.2], seed = 6942067)
print(f"Train Data: {train.count()}, Test Data: {test.count()}")

Train Data: 3451681, Test Data: 863720


In [25]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml import Pipeline

#

country_indexer = StringIndexer(inputCol="Area", outputCol="Area_index")
item_indexer = StringIndexer(inputCol="Item", outputCol="Item_index")

encoder = OneHotEncoder(
    inputCols=["Area_index", "Item_index"],
    outputCols=["Area_vec", "Item_vec"]
)

assembler = VectorAssembler(
    inputCols=["Year", "Area_vec", "Item_vec"],
    outputCol="features"
)

lrModel = LinearRegression(featuresCol="features", labelCol="logValue")
pipeline = Pipeline(stages=[
    country_indexer,
    item_indexer,
    encoder,
    assembler,
    lrModel
])

model = pipeline.fit(train)

predictions = model.transform(test)
predictions.select("Area", "Item", "Year", "logValue", "Prediction").orderBy("Value", ascending=False).show()

+--------------------+--------------------+----+------------------+------------------+
|                Area|                Item|Year|          logValue|        Prediction|
+--------------------+--------------------+----+------------------+------------------+
|               China|Total Merchandise...|2022|22.267184008230664| 19.68832560277643|
|               China|Total Merchandise...|2023|22.202113485222867|19.735070424083034|
|China (excluding ...|Total Merchandise...|2023|22.202113485222867|19.540950192720643|
|               China|Total Merchandise...|2022|22.067924908327797| 19.68832560277643|
|China (excluding ...|Total Merchandise...|2022|22.067924908327797| 19.49420537141404|
|China (excluding ...|Total Merchandise...|2021| 22.05858457374259|19.447460550107422|
|China (excluding ...|Total Merchandise...|2015|21.845186691422974| 19.16699162226776|
|China (excluding ...|Total Merchandise...|2018| 21.84163454694126|19.307226086187598|
|               China|Total Merchandise...|

In [26]:
### Evaluation
lr_model = model.stages[-1]
training_summary = lr_model.summary

print(f"RMSE: {training_summary.rootMeanSquaredError}")
print(f"R2: {training_summary.r2}")

RMSE: 2.39662774304551
R2: 0.5256934953903262
